# E. Emissions Statistics

**Overview.** Final stage: aggregates the per-hour emissions from notebooks C (with-IHS) and D (without-IHS) into the monthly summary statistics and dashboard tables. It sums the time-scaled `…_xhour` pollutant columns by month, country, vessel class/group, operational phase, and fishing-vessel flag; concatenates the with- and without-IHS results (tagged Type A / Type B); maps IHS vessel classes to dashboard vessel types; computes CO₂-equivalent totals (CH₄ × 27, N₂O × 273, GWP100); and builds a fishing-specific breakdown joined to Global Fishing Watch gear types. Outputs feed the `Dashboard Emissions Pacific.xlsx` workbook.

**Series.** Notebook **E** of the five-part series (A–E) estimating ship emissions in Pacific Island EEZs per the Fourth IMO GHG Study 2020:
- **A. AIS Data Prep** — hourly vessel-position panel tagged with EEZ/coast/port and IHS vessel IDs
- **B. IHS Specs** — per-vessel specs, correction factors, and emission factors
- **C. Emissions wIHS** — activity-based emissions for IHS-matched vessels
- **D. Emissions woIHS** — emissions for vessels without an IHS match, via a median-per-type model
- **E. Emissions Statistics** — monthly country / vessel-type / fishing aggregations and dashboard CSVs *(this notebook)*

**Inputs.**
- With-IHS emissions from C: `…/emissions_data/with_ihs/`.
- Without-IHS emissions from D: `…/emissions_data/no_ihs/`.
- Global Fishing Watch effort with gear type: `…/worldbank/GFW_effort_event/*.csv`.
- Intermediate per-method CSVs it writes and re-reads under `…/emissions_data/stats/` (e.g. `Monthly Emissions wIHS …`, `Monthly Emissions no IHS …`).
- Platform S3 root `"s3a://" + os.environ["AWS_WORKING_DIRECTORY_PATH"] + "worldbank/"`.

**Output.** Monthly statistics CSVs under `…/emissions_data/stats/` — combined `Monthly Emissions {yymm}_{yymm}.csv`, the op-phase and fishing variants, plus dashboard download files (`{year} Monthly CO2 by Country Vessel Type.csv`, `{year} Monthly GHG by Country.csv`) generated via `af.create_download_link`. These populate `Dashboard Emissions Pacific.xlsx`.

**Requirements.**
- Runs on the **UN Global Platform** — account required: https://datalab.officialstatistics.org
- PySpark cluster with the platform `ais` library for reading partitions and creating download links; aggregation and the final combine/CO₂-eq steps are in pandas.
- CO₂-equivalent uses GWP100 factors CH₄ = 27, N₂O = 273.

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None) #Show all columns in pandas df
pd.set_option('display.max_rows', 100) #Show all columns in pandas df
#silent future warnings
# pd.set_option('future.no_silent_downcasting', True)
pd.options.display.float_format = "{:,.2f}".format

from IPython.core.interactiveshell import InteractiveShell #allow multiple outputs in one jupyter cell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
import numpy as np

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()


In [4]:
import os
ihs_version = "20250602"
wb_path = "s3a://" + os.environ["AWS_WORKING_DIRECTORY_PATH"] + "worldbank/"
project_path = wb_path + "emissions/Pacific/"
out_path = project_path + "emissions_data/"
temp_path = wb_path + "temp/"
ais_base_path = project_path+"hourly_ais_v2/"
wihs_path = f"{out_path}with_ihs/"


ghg_list= ['_ch4_e', '_co_e','_n2o_e','_nmvoc_e', '_pm10_e', '_pm25_e', '_nox_e','_bc_e','_co2_f','_sox_f','_bc_f']
vessel_type_list = ["Cargo","Tanker","Passenger","Fishing","Tug","Others"]

# All

_Defines the list of pollutant columns to aggregate._


In [5]:
ghg_list= ['_ch4_e', '_co_e','_n2o_e','_nmvoc_e', '_pm10_e', '_pm25_e', '_nox_e','_bc_e','_co2_f','_sox_f','_bc_f']

## Emissions w/ IHS

_Reads the with-IHS emissions and aggregates monthly totals and vessel counts by country, vessel class, operational phase, and fishing flag._


In [6]:
year=2026
yymm = "202601"

In [7]:
sdf = spark.read.option("basepath",f"{out_path}with_ihs/").parquet(f"{out_path}with_ihs/year={year}")
sdf.printSchema()

/opt/spark/python/lib/pyspark.zip/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.


Closing down clientserver connection
root
 |-- LRIMOShipNo: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- mmsi: integer (nullable = true)
 |-- Country_sub: string (nullable = true)
 |-- vessel_id: string (nullable = true)
 |-- imo: integer (nullable = true)
 |-- vessel_name: string (nullable = true)
 |-- vessel_type: string (nullable = true)
 |-- length: double (nullable = true)
 |-- width: double (nullable = true)
 |-- ISO3: string (nullable = true)
 |-- eez_flag: boolean (nullable = true)
 |-- coast_flag: boolean (nullable = true)
 |-- Port_name: string (nullable = true)
 |-- port_flag: boolean (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- sog: double (nullable = true)
 |-- draught: double (nullable = true)
 |-- ihs_matchtype: integer (nullable = true)
 |-- eez_flag_next: boolean (nullable = true)
 |-- datetime_next: timestamp (nullable = true)
 |-- draught_nozero: 

In [8]:
sdf.select(F.max("month"), F.max("date")).show()

+----------+----------+
|max(month)| max(date)|
+----------+----------+
|         1|2026-01-31|
+----------+----------+



In [9]:
sdf_agg = sdf \
.groupBy("year","month","_vessel_class","_w_fishing","Country").agg(*[F.sum(F.col(x+"_xhour")).alias(x) for x in ghg_list],
                                                                    F.countDistinct("vessel_id").alias("count_vessel")
                                                                    )
df = sdf_agg.toPandas()
df.to_csv(f"{out_path}stats/Monthly Emissions wIHS {year}01_{yymm}.csv", index=False)

Found credentials from IAM Role: eksctl-sparky-mc-sparkface-nodegr-NodeInstanceRole-15Y58CLME2WUS


In [10]:
df['_co2_f'].sum() / 10**6

731598.1659414637

In [11]:
df.groupby("Country")['_co2_f'].sum() / 10**6

Country
Cook Islands        14,151.21
Fiji                17,057.00
Kiribati            44,062.32
Marshall Islands    27,763.72
Micronesia         162,771.87
Nauru                4,330.53
Niue                 1,832.14
Palau               38,015.54
Papua New Guinea   375,143.40
Samoa                1,418.16
Solomon Islands     21,229.41
Tokelau              1,755.95
Tonga                6,017.80
Tuvalu               4,182.52
Vanuatu             11,866.58
Name: _co2_f, dtype: float64

In [12]:
sdf_agg = sdf \
.groupBy("year","month","_vessel_class","_op_phase","_w_fishing","Country").agg(*[F.sum(F.col(x+"_xhour")).alias(x) for x in ghg_list],
                                                                    F.countDistinct("vessel_id").alias("count_vessel")
                                                                    )
df = sdf_agg.toPandas()
df.to_csv(f"{out_path}stats/Monthly Emissions opphase wIHS {year}01_{yymm}.csv", index=False)

In [13]:
sdf.select("eez_flag").distinct().show()

+--------+
|eez_flag|
+--------+
|    true|
+--------+



## W/o IHS

_Same monthly aggregation for the without-IHS emissions (EEZ only)._


In [14]:
year=2026
yymm = "202601"

In [15]:
spark.read.option("basepath",f"{out_path}no_ihs/").parquet(f"{out_path}no_ihs/year={year}").select("eez_flag").distinct().show()

+--------+
|eez_flag|
+--------+
|    true|
|   false|
+--------+



In [16]:
sdf = spark.read.option("basepath",f"{out_path}no_ihs/").parquet(f"{out_path}no_ihs/year={year}").filter(F.col("eez_flag"))
sdf.printSchema()

root
 |-- _vessel_group_ais: string (nullable = true)
 |-- _op_phase: string (nullable = true)
 |-- date: date (nullable = true)
 |-- mmsi: integer (nullable = true)
 |-- Country_sub: string (nullable = true)
 |-- vessel_id: string (nullable = true)
 |-- imo: integer (nullable = true)
 |-- vessel_name: string (nullable = true)
 |-- vessel_type: string (nullable = true)
 |-- length: double (nullable = true)
 |-- width: double (nullable = true)
 |-- ISO3: string (nullable = true)
 |-- eez_flag: boolean (nullable = true)
 |-- coast_flag: boolean (nullable = true)
 |-- Port_name: string (nullable = true)
 |-- port_flag: boolean (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- sog: double (nullable = true)
 |-- draught: double (nullable = true)
 |-- LRIMOShipNo: integer (nullable = true)
 |-- ihs_matchtype: integer (nullable = true)
 |-- eez_flag_next: boolean (nullable = true)
 |-- datetime_ne

In [17]:
sdf.select(F.max("month"), F.max("date")).show()

+----------+----------+
|max(month)| max(date)|
+----------+----------+
|         1|2026-01-31|
+----------+----------+



In [18]:
sdf.show(n=1, truncate=False, vertical=True)

-RECORD 0--------------------------------
 _vessel_group_ais | Others              
 _op_phase         | At Berth            
 date              | 2026-01-28          
 mmsi              | 218016420           
 Country_sub       | Fiji                
 vessel_id         | mmsi-218016420      
 imo               | 0                   
 vessel_name       | 2-CAPITANS          
 vessel_type       | Not Available       
 length            | 0.0                 
 width             | 0.0                 
 ISO3              | FJI                 
 eez_flag          | true                
 coast_flag        | true                
 Port_name         | Suva Harbor         
 port_flag         | true                
 datetime          | 2026-01-28 23:00:00 
 longitude         | 178.42659833        
 latitude          | -18.122445          
 sog               | 0.0                 
 draught           | 0.0                 
 LRIMOShipNo       | NULL                
 ihs_matchtype     | NULL         

In [19]:
sdf_agg = sdf \
.groupBy("year","month","_vessel_group_ais","_w_fishing","Country").agg(*[F.sum(F.col(x+"_xhour")).alias(x) for x in ghg_list],
                                                                        F.countDistinct("vessel_id").alias("count_vessel"))
df = sdf_agg.toPandas()
df.to_csv(f"{out_path}stats/Monthly Emissions no IHS {year}01_{yymm}.csv", index=False)

In [20]:
df

,year,month,_vessel_group_ais,_w_fishing,Country,_ch4_e,_co_e,_n2o_e,_nmvoc_e,_pm10_e,_pm25_e,_nox_e,_bc_e,_co2_f,_sox_f,_bc_f,count_vessel
0,2026,1,Tanker,False,Papua New Guinea,"53,108.49","1,594,198.85","147,635.85","2,984,989.22","2,376,572.92","2,186,447.08","65,099,999.77",0.00,"1,465,003,026.82","10,945,321.11","55,599.98",17
1,2026,1,Cargo,False,Solomon Islands,"21,201.76","633,386.25","64,840.81","1,322,074.16","1,727,226.65","1,589,048.52","30,679,748.36",0.00,"632,861,737.52","10,250,120.40","43,673.47",12
2,2026,1,Fishing,False,Palau,"3,262.34","68,963.35","6,524.81","189,052.49","46,025.46","42,343.42","2,315,640.47",0.00,"61,004,532.13","26,040.94","421,394.82",59
3,2026,1,Others,True,Vanuatu,"2,085.42","61,224.77","6,372.04","117,188.88","42,619.51","39,209.95","2,270,915.54",0.00,"67,577,846.71","36,015.49","12,337.47",3
4,2026,1,Fishing,True,Kiribati,"1,485.99","39,038.10","4,294.19","86,113.38","28,255.35","25,994.93","1,517,230.25",0.00,"45,049,398.60","19,230.19","12,642.52",2
5,2026,1,Passenger,False,Kiribati,"2,196.75","64,649.40","6,465.32","125,241.73","47,343.69","43,556.20","2,369,487.62",0.00,"70,072,904.36","33,540.76","13,903.76",2
6,2026,1,Passenger,False,Tonga,"1,380.77","40,636.31","4,068.02","78,716.28","29,787.20","27,404.23","1,490,813.11",0.00,"44,109,149.99","21,115.12","8,679.88",1
7,2026,1,Fishing,True,Fiji,"2,180.15","57,005.43","6,246.04","126,339.98","41,158.76","37,866.06","2,207,082.18",0.00,"65,380,680.33","27,908.99","24,019.38",7
8,2026,1,Others,True,Palau,151.22,"4,443.59",462.86,"8,496.88","3,089.91","2,842.72","164,950.96",0.00,"4,910,484.83","2,612.45",889.40,1
9,2026,1,Others,True,Micronesia,"1,720.12","50,545.84","5,265.07","96,652.01","35,147.74","32,335.92","1,876,317.13",0.00,"55,856,764.97","29,716.64","10,116.97",2


In [21]:
sdf_agg = sdf \
.groupBy("year","month","_vessel_group_ais","_op_phase","_w_fishing","Country").agg(*[F.sum(F.col(x+"_xhour")).alias(x) for x in ghg_list],
                                                                        F.countDistinct("vessel_id").alias("count_vessel"))
df = sdf_agg.toPandas()
df.to_csv(f"{out_path}stats/Monthly Emissions opphase no IHS {year}01_{yymm}.csv", index=False)

## combine

_Concatenates with- and without-IHS stats (Type A/B), maps vessel classes to dashboard types, and computes CO₂-equivalent totals._


In [22]:
vessel_class_map = {
'Refrigerated cargo':'General cargo',
'Yacht':'Others',
'Ferry-pax only':'Passenger',
'Service - other':'Others',
'Container':'Container',
'Oil tanker':'Tanker',
'Vehicle':'General cargo',
'Offshore':'Others',
'Miscellaneous - fishing':'Fishing',
'General cargo':'General cargo',
'Ferry-RoPax':'Passenger',
'Bulk carrier':'Bulk Carrier',
'Cruise':'Passenger',
'Chemical tanker':'Tanker',
'Ro-Ro':'Passenger',
'Non-Propelled / Non Ship Structure':'Others',
'Liquified gas tanker':'Tanker',
'Service - tug':'Others',
'Other liquids tanker':'Tanker',
'Miscellaneous - other':'Others'
}

vessel_group = {
'General cargo':'Cargo',
'Passenger':'Passenger',
'Others':'Others',
'Container':'Cargo',
'Tanker':'Tanker',
'Fishing':'Fishing',
'Bulk Carrier':'Cargo',
'Others':'Others'}

In [23]:
df = pd.concat([pd.read_csv(f"{out_path}stats/Monthly Emissions wIHS 202601_202601.csv").assign(Type="A"),
           pd.read_csv(f"{out_path}stats/Monthly Emissions no IHS 202601_202601.csv").assign(Type="B")])

df['_vessel_group_ais'] = df['_vessel_group_ais'].fillna(df['_vessel_class'].map(vessel_class_map)).replace("Cargo","General cargo").replace("Tug","Others")
df['_vessel_group'] = df['_vessel_group_ais'].map(vessel_group)
df['_vessel_class'] = df['_vessel_class'].fillna("no IHS")

df.rename(columns={'_vessel_group':'Vessel Type',
                   '_vessel_group_ais':'Vessel Subtype',
                   '_vessel_class':'Vessel Class (IMO)'}, inplace=True)

df['ch4_co2eq'] = df['_ch4_e'] * 27
df['n2o_co2_eq'] = df['_n2o_e'] * 273
df['co2eq_total'] = df['_co2_f'] + df['ch4_co2eq']  + df['n2o_co2_eq']
df['bc'] = df['_bc_e'] + df['_bc_f']

df.to_csv(f"{out_path}stats/Monthly Emissions 202601_202601.csv")

In [24]:
df = pd.concat([pd.read_csv(f"{out_path}stats/Monthly Emissions wIHS 202601_202601.csv").assign(Type="A"),
           pd.read_csv(f"{out_path}stats/Monthly Emissions no IHS 202601_202601.csv").assign(Type="B")])

df['_vessel_group_ais'] = df['_vessel_group_ais'].fillna(df['_vessel_class'].map(vessel_class_map)).replace("Cargo","General cargo").replace("Tug","Others")
df['_vessel_group'] = df['_vessel_group_ais'].map(vessel_group)
df['_vessel_class'] = df['_vessel_class'].fillna("no IHS")

df.rename(columns={'_vessel_group':'Vessel Type',
                   '_vessel_group_ais':'Vessel Subtype',
                   '_vessel_class':'Vessel Class (IMO)'}, inplace=True)

df['ch4_co2eq'] = df['_ch4_e'] * 27
df['n2o_co2_eq'] = df['_n2o_e'] * 273
df['co2eq_total'] = df['_co2_f'] + df['ch4_co2eq']  + df['n2o_co2_eq']
df['bc'] = df['_bc_e'] + df['_bc_f']

df.to_csv(f"{out_path}stats/Monthly Emissions opphase 202601_202601.csv")

## w fishing

_Builds the fishing-specific breakdown joined to GFW gear types, with PNA grouping and CO₂-eq._


In [25]:
from pyspark.sql.types import *

In [26]:
gfw = spark.read.option("header",True).csv(f"{wb_path}GFW_effort_event/*.csv") \
.select(F.col("mmsi").cast(IntegerType()),
        F.col("Country").alias("Country_sub"),
        F.to_date("date").alias("date"), 
        "hours",
        "geartype")

In [36]:
sdf = spark.read.option("basepath",f"{out_path}with_ihs").parquet(f"{out_path}with_ihs/year=2026/").filter(F.col("_w_fishing")).join(gfw, on=['date','mmsi','Country_sub'], how="inner") \
.fillna("unknown",subset=['geartype'])

In [38]:
sdf_agg = sdf \
.select("date",F.year("date").alias("year"), F.month("date").alias("month"),"geartype","hours","Country", "vessel_id").distinct() \
.groupBy("year","month","geartype","Country").agg(F.count("vessel_id").alias("days_mmsi"), F.sum("hours").alias("hours"),
                                                  F.countDistinct("vessel_id").alias("count_vessel"))

df2 = sdf_agg.toPandas()

In [39]:
sdf_agg = sdf.groupBy("year","month","geartype","Country").agg(*[F.sum(F.col(x+"_xhour")).alias(x) for x in ghg_list])
df = sdf_agg.toPandas()

In [40]:
df_wihs = df.merge(df2, on=['year','month','geartype','Country'])

In [41]:
sdf = spark.read.option("basepath",f"{out_path}no_ihs/").parquet(f"{out_path}no_ihs/year=2026/").filter(F.col("_w_fishing") & F.col("eez_flag")).join(gfw, on=['date','mmsi','Country_sub'], how="inner") \
.fillna("unknown",subset=['geartype'])

sdf_agg = sdf.groupBy("year","month","geartype","Country").agg(*[F.sum(F.col(x+"_xhour")).alias(x) for x in ghg_list])
df = sdf_agg.toPandas()

sdf_agg = sdf \
.select("date",F.year("date").alias("year"), F.month("date").alias("month"),"vessel_id","geartype","hours","Country").distinct() \
.groupBy("year","month","geartype","Country").agg(F.count("vessel_id").alias("days_mmsi"), F.sum("hours").alias("hours"),
                                                 F.countDistinct("vessel_id").alias("count_vessel"))

df2 = sdf_agg.toPandas()
df_woihs = df.merge(df2, on=['year','month','geartype','Country'])

In [42]:
df_woihs

,year,month,geartype,Country,_ch4_e,_co_e,_n2o_e,_nmvoc_e,_pm10_e,_pm25_e,_nox_e,_bc_e,_co2_f,_sox_f,_bc_f,days_mmsi,hours,count_vessel
0,2026,1,DRIFTING_LONGLINES,Marshall Islands,"19,171.63","551,457.97","58,003.64","1,083,835.72","386,216.92","355,319.56","20,638,010.18",0.00,"613,988,419.27","314,757.05","128,373.37",112,"1,321.64",6
1,2026,1,INCONCLUSIVE,Palau,151.22,"4,443.59",462.86,"8,496.88","3,089.91","2,842.72","164,950.96",0.00,"4,910,484.83","2,612.45",889.40,12,211.54,1
2,2026,1,POLE_AND_LINE,Kiribati,"9,748.81","287,417.61","29,834.08","559,753.76","317,713.03","292,295.99","11,325,496.61",0.00,"311,464,274.32","1,073,595.56","49,920.45",43,536.25,3
3,2026,1,FISHING,Papua New Guinea,5.03,122.59,11.78,291.62,80.87,74.40,"4,177.37",0.00,"115,621.48",49.36,145.49,1,3.52,1
4,2026,1,TUNA_PURSE_SEINES,Solomon Islands,"1,266.58","32,597.09","3,550.15","73,398.52","23,474.13","21,596.20","1,254,705.61",0.00,"36,967,890.46","15,780.45","27,526.07",41,480.75,2
5,2026,1,INCONCLUSIVE,Fiji,"3,956.61","114,167.73","11,993.51","223,494.14","79,883.01","73,492.37","4,268,269.46",0.00,"127,003,214.46","65,443.65","25,485.19",20,194.44,1
6,2026,1,DRIFTING_LONGLINES,Fiji,"40,137.07","1,174,251.12","122,475.52","2,257,910.19","817,836.55","752,409.63","43,634,959.68",0.00,"1,298,545,045.31","686,712.47","245,465.13",161,"2,500.54",9
7,2026,1,POLE_AND_LINE,Papua New Guinea,604.88,"17,774.36","1,851.45","33,987.52","12,359.64","11,370.87","659,803.83",0.00,"19,641,939.33","10,449.81","3,557.61",2,7.09,1
8,2026,1,FISHING,Kiribati,"8,298.16","243,842.00","25,399.64","466,266.30","169,558.88","155,994.17","9,051,683.76",0.00,"269,462,855.19","143,358.29","48,806.03",24,356.88,2
9,2026,1,FISHING,Fiji,55.10,"1,449.78",159.59,"3,193.15","1,049.71",965.73,"56,385.02",0.00,"1,675,102.24",715.05,413.41,1,3.40,1


In [43]:
gear_map = {'SET_LONGLINES':['set longlines','longlines'],
'TUNA_PURSE_SEINES':['tuna purse seines','purse seines'],
'DRIFTING_LONGLINES':['drifting longlines','longlines'],
'INCONCLUSIVE':['inconclusive','unknown'],
'PURSE_SEINES':['purse seines','purse seines'],
'OTHER_PURSE_SEINES':['purse seines','purse seines'],
'GEAR':['other','others'],
'FISHING':['fishing','unknown'],
'OTHER':['other','others'],
'POLE_AND_LINE':['pole and line','others'],
'FIXED_GEAR':['fixed gear','others'],
'SQUID_JIGGER':['squid jigger','others'],
'TRAWLERS':['trawlers','others'],
'SET_GILLNETS':['set gillnets','others']

           }


pna_map = {
'Kiribati':'PNA Member', 
    'Marshall Islands':'PNA Member', 
    'Solomon Islands':'PNA Member', 
    'Nauru':'PNA Member',
    'Vanuatu':'Others', 
    'Fiji':'Others', 
    'Tokelau':'PNA Member', 
    'Tuvalu':'PNA Member', 
    'Papua New Guinea':'Others',
    'Micronesia':'PNA Member', 
    'Palau':'PNA Member', 
    'Tonga':'Others', 
    'Samoa':'Others'
}


In [44]:
df3 = pd.concat([df_wihs.assign(Type="A"), df_woihs.assign(Type="B")], ignore_index=True).rename(columns={'count':'days_mmsi'})
df3['ch4_co2eq'] = df3['_ch4_e'] * 27
df3['n2o_co2_eq'] = df3['_n2o_e'] * 273
df3['co2eq_total'] = df3['_co2_f'] + df3['ch4_co2eq']  + df3['n2o_co2_eq']
df3['bc'] = df3['_bc_e'] + df3['_bc_f']
df3['PNA'] = df3['Country'].map(pna_map)



df3[['Gear Type','Gear Category']] = pd.DataFrame.from_dict(df3['geartype'].map(gear_map).to_dict()).T

In [45]:
df3.to_csv(f"{out_path}stats/Monthly Emissions fishing 202601_202601.csv", index=False)

# Dashboard

_Reads the combined monthly stats and produces the dashboard download CSVs (CO₂ by country/vessel type, GHG by country)._


In [46]:
df = pd.read_csv(f"{out_path}stats/Monthly Emissions 202601_202601.csv", index_col=0)
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 281 entries, 0 to 71
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   year                281 non-null    int64  
 1   month               281 non-null    int64  
 2   Vessel Class (IMO)  281 non-null    object 
 3   _w_fishing          281 non-null    bool   
 4   Country             281 non-null    object 
 5   _ch4_e              281 non-null    float64
 6   _co_e               281 non-null    float64
 7   _n2o_e              281 non-null    float64
 8   _nmvoc_e            281 non-null    float64
 9   _pm10_e             281 non-null    float64
 10  _pm25_e             281 non-null    float64
 11  _nox_e              281 non-null    float64
 12  _bc_e               281 non-null    float64
 13  _co2_f              281 non-null    float64
 14  _sox_f              281 non-null    float64
 15  _bc_f               281 non-null    float64
 16  count_ves

In [47]:
df.head()

,year,month,Vessel Class (IMO),_w_fishing,Country,_ch4_e,_co_e,_n2o_e,_nmvoc_e,_pm10_e,_pm25_e,_nox_e,_bc_e,_co2_f,_sox_f,_bc_f,count_vessel,Type,Vessel Subtype,Vessel Type,ch4_co2eq,n2o_co2_eq,co2eq_total,bc
0,2026,1,Vehicle,False,Kiribati,"145,264.88","797,078.25","66,851.34","1,316,535.89","1,422,251.44","1,308,471.33","35,438,647.39",571.31,"582,305,481.92","7,723,738.79","28,118.06",7,A,General cargo,Cargo,"3,922,151.63","18,250,415.53","604,478,049.08","28,689.36"
1,2026,1,Miscellaneous - fishing,False,Marshall Islands,"53,993.89","1,355,069.06","160,658.67","3,167,424.68","1,424,612.74","1,310,643.72","59,229,677.51",0.00,"1,590,471,340.19","3,254,756.02","385,801.65",104,A,Fishing,Fishing,"1,457,834.95","43,859,816.29","1,635,788,991.43","385,801.65"
2,2026,1,Yacht,False,Palau,766.79,"22,926.88","2,353.73","43,661.61","14,897.74","13,705.92","851,957.53",0.00,"23,660,635.77","10,099.99","63,529.74",3,A,Others,Others,"20,703.44","642,568.61","24,323,907.82","63,529.74"
3,2026,1,Bulk carrier,False,Palau,"666,965.36","15,869,339.34","1,617,134.04","32,888,923.15","43,251,902.77","39,791,750.55","774,576,582.06",421.11,"15,843,628,902.31","257,200,851.21","1,080,381.43",246,A,Bulk Carrier,Cargo,"18,008,064.85","441,477,591.60","16,303,114,558.77","1,080,802.54"
4,2026,1,Ferry-RoPax,False,Palau,113.47,"3,313.23",363.09,"6,575.41","9,243.42","8,503.95","119,140.22",0.00,"3,953,020.92","64,527.37","2,810.55",1,A,Passenger,Passenger,"3,063.61","99,124.66","4,055,209.18","2,810.55"


In [48]:
stats = df.groupby(['year','month','Vessel Type','Country']).agg({'count_vessel':'sum',
                                                          'co2eq_total':'sum'})
stats['CO2eq'] = (stats['co2eq_total'] / 10**6).round()
stats.drop(columns=['co2eq_total'], inplace=True)
stats.rename(columns={'count_vessel':'Vessel'}, inplace=True)
stats

Vessel      CO2eq
year month Vessel Type Country                            
2026 1     Cargo       Cook Islands          55  10,846.00
                       Fiji                  59   8,986.00
                       Kiribati             162  32,243.00
                       Marshall Islands     245  25,113.00
                       Micronesia           977 133,954.00
                       Nauru                 43   1,683.00
                       Niue                  15   1,269.00
                       Palau                469  36,162.00
                       Papua New Guinea    1238 289,871.00
                       Samoa                 16     599.00
                       Solomon Islands      165  16,207.00
                       Tokelau               16   1,172.00
                       Tonga                 25   3,577.00
                       Tuvalu                28   3,117.00
                       Vanuatu               48   4,105.00
           Fishing     Cook Islands          38     696.00
                       Fiji                 127     737.00
                       Kiribati             284   8,964.00
                       Marshall Islands     166   2,791.00
                       Micronesia           219   6,317.00
                       Nauru                 60   1,786.00
                       Niue                   5       5.00
                       Palau                 81     312.00
                       Papua New Guinea     206   7,696.00
                       Samoa                 14      52.00
                       Solomon Islands      116   1,693.00
                       Tokelau               12      62.00
                       Tonga                 19     120.00
                       Tuvalu                47     647.00
                       Vanuatu               56     809.00
           Others      Cook Islands          35   1,481.00
                       Fiji                 338  20,776.00
                       Kiribati             244  18,809.00
                       Marshall Islands      43   2,356.00
                       Micronesia           111   4,250.00
                       Nauru                 16     118.00
                       Niue                   2      35.00
                       Palau                 54     753.00
                       Papua New Guinea     427  11,571.00
                       Samoa                 11      73.00
                       Solomon Islands       67   2,475.00
                       Tokelau               11     450.00
                       Tonga                  8      87.00
                       Tuvalu                15     304.00
                       Vanuatu               46   1,396.00
           Passenger   Cook Islands          10   1,892.00
                       Fiji                  28   5,812.00
                       Kiribati               6   1,263.00
                       Marshall Islands       8     102.00
                       Micronesia            16     268.00
                       Nauru                  2       0.00
                       Niue                   4     457.00
                       Palau                 10      47.00
                       Papua New Guinea      42   2,813.00
                       Samoa                  8     627.00
                       Solomon Islands        6   1,178.00
                       Tokelau                1      21.00
                       Tonga                 10   2,026.00
                       Tuvalu                 1       0.00
                       Vanuatu               19   6,549.00
           Tanker      Cook Islands          11     992.00
                       Fiji                  15   1,953.00
                       Kiribati              22   3,211.00
                       Marshall Islands      57   2,452.00
                       Micronesia           208  41,434.00
                       Nauru                 16   1,105.00
                       Niue  

In [49]:
from ais import functions as af

In [50]:
year=2026
filename = f"{year} Monthly CO2 by Country Vessel Type.csv"
af.create_download_link(stats, filename, filename)

In [51]:
stats =  df.groupby(['year','month','Country'])[[
    'bc', '_sox_f','_co2_f', '_nox_e', '_pm25_e', '_pm10_e','_nmvoc_e', '_n2o_e', '_co_e',  '_ch4_e'
        ]].sum()
stats.rename(columns={'bc':'BC',
                      '_sox_f':'SOX',
                      '_co2_f':'CO2',
                      '_nox_e':'NOx',
                      '_pm25_e':'PM2.5',
                      '_pm10_e':'PM10',
                      '_nmvoc_e':'NMVOC',
                      '_n2o_e':'N2O',
                      '_co_e':'CO',
                      '_ch4_e':'CH4'
                     }, inplace=True)
filename = f"{year} Monthly GHG by Country.csv"
af.create_download_link(stats, filename, filename)